# Exercise 10 — Multi-factor models

Lecture 10 made two claims. That a factor is *constructed* rather than discovered, and
that the choice of benchmark decides whether a return is alpha or style. This lab does
both, on the one anomaly the CAPM fails most visibly: the flat security market line.

You will sort on beta, build your own betting-against-beta factor out of the failure,
and then ask whether that factor is anything new.

**Data.** Ten value-weighted portfolios formed on beta, the Fama-French five factors,
the momentum factor, and the AQR BAB factors — all monthly, July 1963 to June 2026.
Sources: [Kenneth R. French Data
Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html) and
[AQR Capital Management](https://www.aqr.com/Insights/Datasets).

**How to work with this notebook.** The task text is here and on the exercise sheet.
Start with **Runtime → Restart and run all** so the imports and the data are in place,
then work through the tasks in order. Task 4 is optional and comes with its code
already written.

In [ ]:
!wget -q https://raw.githubusercontent.com/KroeTiA/Investments/main/exercise_utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from matplotlib.ticker import PercentFormatter

from exercise_utils import FHNW, ASSET_CYCLE, setup_style, save_results
setup_style()

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")


def as_percent(df, cols):
    # display copy with the named columns scaled to percentage points
    out = df.copy()
    out[cols] = out[cols] * 100
    return out.to_string(float_format=lambda v: f"{v:8.2f}")

### The data

The French files are committed verbatim, copyright lines and all, so the cleaning
happens here in front of you rather than somewhere upstream. Each file stacks several
blocks on top of each other — value-weighted monthly, equal-weighted monthly, then the
annual versions — so we start at the header row of the block we want and stop at the
first row that is not a `YYYYMM` date.

Returns arrive in percentage points and are converted to decimals. Missing values are
coded `-99.99` or `-999`.

In [ ]:
BASE = ("https://raw.githubusercontent.com/KroeTiA/Investments/main/"
        "Exercise_10/data/")


def read_french(url, skiprows):
    # read one block of a Kenneth French CSV into monthly decimal returns
    df = pd.read_csv(url, skiprows=skiprows)
    df = df.rename(columns={df.columns[0]: "date"})
    df["date"] = df["date"].astype(str).str.strip()
    is_month = df["date"].str.fullmatch(r"\d{6}")
    stop = (~is_month).values.argmax() if (~is_month).any() else len(df)
    df = df.iloc[:stop]
    df = df.set_index(pd.PeriodIndex(df["date"], freq="M")).drop(columns="date")
    df.columns = [c.strip() for c in df.columns]
    return df.astype(float).replace([-99.99, -999.0], np.nan) / 100


# skiprows points at the header line of the block we want
ports = read_french(BASE + "Portfolios_Formed_on_BETA.csv", 15)   # VW, monthly
ff5 = read_french(BASE + "F-F_Research_Data_5_Factors_2x3.csv", 4)
mom = read_french(BASE + "F-F_Momentum_Factor.csv", 13)

DEC = ["Lo 10", "Dec 2", "Dec 3", "Dec 4", "Dec 5",
       "Dec 6", "Dec 7", "Dec 8", "Dec 9", "Hi 10"]

data = ports[DEC].join(ff5, how="inner").join(mom, how="inner").dropna()
rx = data[DEC].sub(data["RF"], axis=0)      # excess returns of the ten deciles
mkt = data["Mkt-RF"]                        # the market factor

print(f"{len(data)} months, {data.index[0]} to {data.index[-1]}")
print(f"columns available as factors: {list(ff5.columns) + list(mom.columns)}")
data.head(3)

## Task 1 — Does the CAPM survive a sort on beta?

Every June, French sorts the whole US market into ten value-weighted portfolios by
estimated beta. `rx` holds their monthly excess returns. If the CAPM held, the ten
average excess returns would sit on a straight line through the origin whose slope is
the market risk premium.

**(a)** Build one table, `capm`, with a row per decile and these columns: annualised
mean excess return, annualised volatility, annualised Sharpe ratio, the full-sample
beta, the annualised CAPM alpha, and the t-statistic of the monthly alpha.

**(b)** Plot the ten portfolios in beta–mean-return space. Draw the theoretical
security market line through the origin and the market portfolio, and fit a line
through the ten points. Report the fitted intercept and slope next to their
theoretical values.

*Deliverable: the table, the figure, and the two pairs of numbers — intercept fitted
against 0, slope fitted against the market risk premium.*

Then say which alphas you would call economically large, and whether their signs match
what the lecture predicted.

In [ ]:
# SOLUTION
# STUB: for each decile, regress its excess return on the market excess return and
# STUB: collect mean, volatility, Sharpe, beta, alpha (annualised) and t(alpha)
rows = {}
for p in DEC:                                                              # KEEP
    res = sm.OLS(rx[p], sm.add_constant(mkt)).fit()
    rows[p] = {
        "mean": rx[p].mean() * 12,
        "sd": rx[p].std() * np.sqrt(12),
        "sr": rx[p].mean() / rx[p].std() * np.sqrt(12),
        "beta": res.params.iloc[1],
        "alpha": res.params.iloc[0] * 12,
        "t(alpha)": res.tvalues.iloc[0],
    }

capm = pd.DataFrame(rows).T
print("mean, sd and alpha in % p.a.\n")                                    # KEEP
print(as_percent(capm, ["mean", "sd", "alpha"]))                           # KEEP

rp_market = mkt.mean() * 12                                                # KEEP
print(f"\nmarket risk premium {rp_market:.2%} p.a., "
      f"volatility {mkt.std() * np.sqrt(12):.2%}, "
      f"Sharpe {mkt.mean() / mkt.std() * np.sqrt(12):.3f}")

In [ ]:
# SOLUTION
# STUB: fit the empirical SML by regressing the ten mean returns on the ten betas
sml = sm.OLS(capm["mean"], sm.add_constant(capm["beta"])).fit()
print(f"fitted   intercept {sml.params.iloc[0]:+.2%} (t {sml.tvalues.iloc[0]:.2f}), "
      f"slope {sml.params.iloc[1]:+.2%} (t {sml.tvalues.iloc[1]:.2f}), "
      f"R2 {sml.rsquared:.3f}")
print(f"theory   intercept  0.00%,          slope {rp_market:+.2%}")

# STUB: scatter the ten portfolios, then add the theoretical and the fitted line
fig1, ax = plt.subplots(figsize=(7, 4.5))                                  # KEEP
grid = np.linspace(0, 1.75, 50)                                            # KEEP
ax.plot(grid, rp_market * grid, color=FHNW["navy"], lw=1.6,
        label="theoretical SML")
ax.plot(grid, sml.params.iloc[0] + sml.params.iloc[1] * grid,
        color=FHNW["red"], lw=1.6, ls="--", label="fitted line")
ax.scatter(capm["beta"], capm["mean"], s=45, color=FHNW["blue"], zorder=3,
           label="beta deciles")
ax.scatter([1.0], [rp_market], s=90, marker="*", color=FHNW["green"],
           zorder=4, label="market")
for p, off in [("Lo 10", (-4, 10)), ("Hi 10", (8, -4))]:
    ax.annotate(p, (capm.loc[p, "beta"], capm.loc[p, "mean"]),
                textcoords="offset points", xytext=off, fontsize=9)
ax.set_xlabel("beta")                                                      # KEEP
ax.set_ylabel("mean excess return, p.a.")                                  # KEEP
ax.yaxis.set_major_formatter(PercentFormatter(1.0))                        # KEEP
ax.legend(frameon=False)                                                   # KEEP
plt.show()                                                                 # KEEP

<!-- solution -->
### What Task 1 shows

The Sharpe ratio falls from **0.55** in the lowest-beta decile to **0.31** in the
highest, while the market sits at **0.47**. Bearing more market risk was not rewarded:
it was punished. The alphas run **+2.37 % p.a. (t = 2.41)** at the bottom and
**−2.71 % p.a. (t = −1.52)** at the top, and the pattern in between is close to
monotone.

The cross-sectional fit is the sharper statement. The empirical line has intercept
**+4.24 %** where theory says zero, and slope **+3.66 %** where theory says the market
risk premium of **7.21 %**. The line is real — beta does sort average returns, R² is
0.67 — but it is **roughly half as steep as it should be, and lifted off the origin**.
This is Black, Jensen and Scholes (1972), still there sixty years later.

Two things students get wrong here. The first is regressing *total* returns on the
market excess return; both sides must be in excess of the risk-free rate or the
intercept absorbs the risk-free rate. The second is reading the flat line as "beta is
not priced". It is priced — just not at the rate the CAPM demands. A model can be
directionally right and quantitatively wrong, and that is exactly the gap a second
factor is invented to fill.

## Task 2 — Build a factor out of the failure

The lecture's claim: a factor is not discovered in the data, it is constructed.
Task 1 handed you a characteristic that sorts average returns the wrong way. Turn it
into a portfolio.

**(a)** The obvious attempt first. Go long one dollar of the lowest-beta decile and
short one dollar of the highest. Report its mean excess return, volatility, beta and
alpha.

**(b)** Now make it market-neutral. Lever the low-beta leg up and the high-beta leg
down so that each carries a beta of exactly one before you subtract: hold
$1/\beta_L$ in `Lo 10` and $1/\beta_H$ in `Hi 10`, with the difference borrowed or lent
at the risk-free rate. Call the result `bab`. Confirm its beta is zero and report its
Sharpe ratio and alpha.

**(c)** Plot the compounded excess return of `bab` against the market.

*Deliverable: the two weights, a printed comparison of the naive and the rescaled
portfolio on mean, volatility, Sharpe, beta and alpha, and the figure.*

Then answer the question the comparison raises: the naive portfolio has a **negative**
average return and a **positive** alpha. How can both be true at once?

In [ ]:
# SOLUTION
# STUB: the naive long-short, then the two beta-neutralising weights, then bab
naive = rx["Lo 10"] - rx["Hi 10"]

w_long = 1 / capm.loc["Lo 10", "beta"]
w_short = 1 / capm.loc["Hi 10", "beta"]
bab = w_long * rx["Lo 10"] - w_short * rx["Hi 10"]

print(f"long  {w_long:.3f} x Lo 10   (beta {capm.loc['Lo 10', 'beta']:.3f})")
print(f"short {w_short:.3f} x Hi 10   (beta {capm.loc['Hi 10', 'beta']:.3f})")
print(f"net exposure to the risk-free asset: {w_short - w_long:+.3f}\n")

# STUB: summarise both portfolios in one table
def summarise(r, label):                                                   # KEEP
    m = mkt.reindex(r.index)          # so the helper also works on a shorter sample
    res = sm.OLS(r, sm.add_constant(m)).fit()
    return pd.Series({
        "mean": r.mean() * 12,
        "sd": r.std() * np.sqrt(12),
        "sr": r.mean() / r.std() * np.sqrt(12),
        "beta": res.params.iloc[1],
        "alpha": res.params.iloc[0] * 12,
        "t(alpha)": res.tvalues.iloc[0],
    }, name=label)


factors = pd.DataFrame([summarise(naive, "naive Lo - Hi"),
                        summarise(bab, "BAB")])
print("mean, sd and alpha in % p.a.\n")
print(as_percent(factors, ["mean", "sd", "alpha"]))
print(f"\nfor reference, the market: mean {mkt.mean() * 12:.2%}, "
      f"volatility {mkt.std() * np.sqrt(12):.2%}, "
      f"Sharpe {mkt.mean() / mkt.std() * np.sqrt(12):.3f}")

In [ ]:
# SOLUTION
# STUB: compound both excess return series and plot them on a log scale
wealth = pd.DataFrame({"BAB": (1 + bab).cumprod(),
                       "market": (1 + mkt).cumprod()})

fig2, ax = plt.subplots(figsize=(7, 4.5))                                  # KEEP
wealth.plot(ax=ax, color=[FHNW["blue"], FHNW["navy"]], lw=1.4)             # KEEP
ax.set_yscale("log")                                                       # KEEP
ax.set_xlabel("")                                                          # KEEP
ax.set_ylabel("compounded excess return, 1 = July 1963")                   # KEEP
ax.legend(frameon=False)                                                   # KEEP
plt.show()                                                                 # KEEP

drawdown = wealth["BAB"] / wealth["BAB"].cummax() - 1
print(f"BAB worst drawdown {drawdown.min():.1%}, trough {drawdown.idxmin()}")

<!-- solution -->
### What Task 2 shows

The two weights are **1.681** on the long leg and **0.624** on the short leg, which
leaves the strategy **1.057 dollars short of the risk-free asset** — it is a levered
position, and that is the point. Frazzini and Pedersen's whole argument is that
investors who cannot or will not borrow bid up high-beta stocks instead, and that
someone willing to borrow can collect the difference.

The comparison is the lesson:

| | mean p.a. | vol p.a. | beta | alpha p.a. | t |
|---|---|---|---|---|---|
| naive Lo − Hi | **−2.18 %** | 24.7 % | **−1.01** | **+5.08 %** | 2.08 |
| BAB | **+5.67 %** | 19.1 % | **0.000** | +5.67 % | 2.34 |

The naive portfolio *lost* money on average and still had a positive alpha, because a
beta of −1.01 means the CAPM required it to lose about 7 % a year. Beating a required
return of −7 % with a realised −2 % is a positive alpha. **Alpha is a residual, not a
return** — the single most useful sentence in this course when reading a factsheet.

Rescaling removes the short market exposure, and once it is gone the alpha and the mean
return are the same number by construction, as are the Sharpe ratio and the information
ratio (both **0.298**).

Two caveats, and they matter for Task 4. The betas used here come from the *full
sample*, so the strategy is market-neutral only with hindsight; a real portfolio must
neutralise on betas estimated from data available at the time. And the figure shows why
nobody holds this alone: the compounded excess return falls **−53.6 %** into
February 2000, when high-beta technology stocks ran away from everything else. A
positive alpha with a t of 2.3 over sixty-three years is not a comfortable ride.

## Task 3 — A two-factor model

A factor model is judged by how well it explains a set of test assets. And ideally, a
new factor is not already captured by the existing, well-known ones.

**(a)** Add `bab` to the market and re-run Task 1's ten regressions, now with two
regressors. Put the two-factor alphas and t-statistics next to the CAPM ones, and
report the mean absolute alpha under each model.

**(b)** Now put `bab` on the left-hand side. Regress it on the market alone, then on
FF3 (`Mkt-RF, SMB, HML`), then on FF5 (adds `RMW, CMA`), then on FF5 plus `Mom`. A
regression of one factor on a set of others is a **spanning test**: it asks whether the
returns of your factor could have been reproduced by holding a combination of theirs.
An alpha that survives means it could not.

*Deliverable: the two tables — alphas and t-statistics per decile under both models,
and alpha, t-statistic, R² and the factor loadings for each of the four models.*

Then: which model kills BAB's alpha, and which single factor does most of the work?
And in (a), which two deciles were always going to lose their alpha, whatever the data
had looked like?

### New this week: regression with several factors

`sm.OLS` takes a DataFrame of regressors exactly as it took a single series. The
intercept is still alpha, but it is now measured against a benchmark built from all of
them, so it answers a different question than the CAPM alpha did.

```python
cols = ["Mkt-RF", "SMB", "HML"]
res = sm.OLS(bab, sm.add_constant(data[cols])).fit()
print(res.params, res.tvalues, res.rsquared)
```

Three things that cost groups time every year. Excess returns belong on **both** sides —
the Fama-French factors already are excess returns, the ten portfolios are not. Returns
are decimals here, so annualise a mean by 12 and a volatility by the square root of 12,
and convert to percent only for display. And a long-short portfolio needs no net
investment, so its return is already an excess return; do not subtract the risk-free
rate from it a second time.

In [ ]:
# SOLUTION
# STUB: regress each decile on [market, bab] and compare the alphas to Task 1's
X2 = pd.concat([mkt, bab.rename("BAB")], axis=1)

rows = {}
for p in DEC:                                                              # KEEP
    res = sm.OLS(rx[p], sm.add_constant(X2)).fit()
    rows[p] = {
        "alpha CAPM": capm.loc[p, "alpha"],
        "t CAPM": capm.loc[p, "t(alpha)"],
        "alpha 2F": res.params.iloc[0] * 12,
        "t 2F": res.tvalues.iloc[0],
        "beta BAB": res.params.iloc[2],
    }

two_factor = pd.DataFrame(rows).T
print("alphas in % p.a.\n")                                                # KEEP
print(as_percent(two_factor, ["alpha CAPM", "alpha 2F"]))                  # KEEP
print(f"\nmean |alpha|   CAPM {two_factor['alpha CAPM'].abs().mean():.2%}"
      f"   Mkt+BAB {two_factor['alpha 2F'].abs().mean():.2%}")

In [ ]:
# the four factor sets, so nobody types column names twice
MODELS = {
    "CAPM": ["Mkt-RF"],
    "FF3": ["Mkt-RF", "SMB", "HML"],
    "FF5": ["Mkt-RF", "SMB", "HML", "RMW", "CMA"],
    "FF5 + MOM": ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom"],
}

In [ ]:
# SOLUTION
# STUB: spanning tests -- regress bab on each of the four factor sets in turn
rows = []
for name, cols in MODELS.items():
    res = sm.OLS(bab, sm.add_constant(data[cols])).fit()
    row = {"alpha": res.params.iloc[0] * 12,
           "t(alpha)": res.tvalues.iloc[0],
           "R2": res.rsquared}
    row.update({c: res.params[c] for c in cols})
    rows.append(pd.Series(row, name=name))

spanning = pd.DataFrame(rows)
print("alpha in % p.a., loadings in units of the factor\n")                # KEEP
print(as_percent(spanning, ["alpha"]))                                     # KEEP

<!-- solution -->
### What Task 3 shows

**(a) The two-factor model does price the deciles better, and part of that is
bookkeeping.** Mean absolute alpha falls from **1.09 % to 0.79 %** a year, and the two
alphas that mattered are gone: `Lo 10` from +2.37 % (t = 2.41) to +0.25 % (t = 0.65),
`Hi 10` from −2.71 % to +0.67 %. But `bab` is a linear combination of exactly those two
portfolios, so their alphas had to collapse — no data could have produced anything
else. The informative part of the table is elsewhere: `Dec 8` and `Dec 9` **acquire**
alphas of +1.67 % and +2.38 % (t = 1.95 and 2.26) that the CAPM did not give them.

This is not a flaw in the exercise, it is the reason Lecture 11 spends its time on test
assets and on the GRS statistic. A factor built from the extremes of a sort will always
price those extremes. The question is what it does to everything else — and here it
tilts the middle of the distribution rather than flattening it.

**(b) The spanning test is where the interesting thing happens.**

| model | alpha p.a. | t | R² |
|---|---|---|---|
| CAPM | +5.67 % | 2.34 | 0.00 |
| FF3 | +3.92 % | 1.90 | 0.29 |
| **FF5** | **+0.22 %** | **0.11** | 0.37 |
| FF5 + MOM | −1.64 % | −0.82 | 0.39 |

Against the market alone, BAB is a 5.7 % anomaly. Against FF5 it is **nothing at all**.
The two factors doing the work are **RMW (+0.57) and CMA (+0.80)**: low-beta stocks are
profitable firms that invest conservatively. Hold the profitability and investment
factors and you have already bought most of what BAB sells. Note also the large
negative SMB loading (−0.74) — low-beta stocks are big, high-beta stocks are small.

The lecture showed a manager whose +3.0 % CAPM alpha became −0.4 % once his style
was priced. This is the same arithmetic applied to a factor rather than a fund: nothing
about the strategy changed, only the benchmark did. Whether that is an unmasking or an
injustice depends on whether RMW and CMA are risks investors must be paid to bear, and
the lecture deliberately left that open.

Keep the word **spanning**. It returns in Lecture 11, where the question becomes
whether a set of factors spans Berkshire Hathaway.

## Task 4 (optional) — Your BAB and AQR's

Task 3 concluded that BAB is spanned by FF5. Before you believe it, check the
construction. AQR publishes the BAB factor from Frazzini and Pedersen (2014), built
from *individual stocks* rather than two decile portfolios, rank-weighted across the
whole cross-section, rebalanced monthly, and neutralised on betas estimated from daily
data available at the time and shrunk towards one.

The code below is written for you. Run it, then read the two tables against each other.
The only question is what the difference between them is evidence of.

In [ ]:
aqr_raw = pd.read_excel(BASE + "Betting_Against_Beta_Equity_Factors_Monthly.xlsx",
                        sheet_name="BAB Factors", skiprows=18,
                        usecols=["DATE", "USA"]).dropna()
aqr = aqr_raw.set_index(pd.PeriodIndex(pd.to_datetime(aqr_raw["DATE"]),
                                       freq="M"))["USA"]

both = pd.concat([bab.rename("ours"), aqr.rename("AQR")], axis=1).dropna()
print(f"overlap {len(both)} months, {both.index[0]} to {both.index[-1]}")
print(f"correlation {both['ours'].corr(both['AQR']):.3f}\n")

print(as_percent(pd.DataFrame([summarise(both["ours"], "ours"),
                               summarise(both["AQR"], "AQR")]),
                 ["mean", "sd", "alpha"]))

rows = []
for name, cols in MODELS.items():
    j = pd.concat([aqr.rename("y"), data[cols]], axis=1).dropna()
    res = sm.OLS(j["y"], sm.add_constant(j[cols])).fit()
    rows.append(pd.Series({"alpha": res.params.iloc[0] * 12,
                           "t(alpha)": res.tvalues.iloc[0],
                           "R2": res.rsquared}, name=name))

print("\nspanning tests for the AQR factor, alpha in % p.a.")
print(as_percent(pd.DataFrame(rows), ["alpha"]))

The two series correlate only **0.42**, and AQR's version earns a Sharpe ratio of
**0.79** against your **0.30** — at *lower* volatility, 11.2 % against 19.1 %.

Read the two ladders side by side and note first what they have in common. Both start
high against the CAPM and fall steeply as factors are added; RMW and CMA take a large
bite out of both. AQR's alpha drops from **+9.38 % to +3.64 %**, losing three fifths of
its size, exactly the direction your own regression found. **The qualitative conclusion
is the same in both.**

What differs is how far the fall goes. Yours crosses zero and ends insignificant; AQR's
stops at +3.64 % with a t-statistic of 2.84 and is still there. Same idea, same
direction, different magnitude — and the magnitude is what decides whether anything is
left to trade.

The difference comes from construction: single stocks instead of two portfolios, the
whole cross-section instead of its tails, ex-ante betas from daily data instead of
full-sample betas, monthly rebalancing instead of none. The lecture said the
construction choices are part of the model. This is the size of that effect.

<!-- solution -->
### Running Task 4 in the walkthrough

If time is short in slot 4, this is the task to spend it on. The pairing to put on the
board is the two spanning ladders side by side:

| model | ours | AQR |
|---|---|---|
| CAPM | +5.67 % (t 2.34) | +9.38 % (t 6.60) |
| FF3 | +3.92 % (t 1.90) | +7.87 % (t 5.75) |
| FF5 | +0.22 % (t 0.11) | +5.20 % (t 4.01) |
| FF5 + MOM | −1.64 % (t −0.82) | +3.64 % (t 2.84) |

The reading to insist on: **the two ladders agree qualitatively**. Both fall steeply,
both fall for the same reason — RMW and CMA — and AQR's loses three fifths of its alpha
on the way down. The disagreement is quantitative: ours crosses zero, theirs stops just
above it. Do not let the room turn this into "one says yes, the other says no".

The mistake worth naming is the other one. A group that concludes "BAB is not a real
factor" from Task 3 alone has drawn a conclusion about their own two-decile
implementation and attributed it to the world — the same error as reading a failed
replication as a refutation. What Task 4 licenses is narrower and more useful: most of
the raw BAB premium is profitability and investment exposure, and whether the remainder
is worth anything depends on how well the strategy is built.

The look-ahead point is worth one minute too. Our beta of exactly 0.0000 is not an
achievement, it is a definition: we neutralised on betas estimated from the same sample
we then measured. AQR's factor has a small negative market beta precisely because it
had to guess.

## Export

Bundle the figures and the tables, in case you want them for the transfer questions or
your own notes.

In [ ]:
# SOLUTION
# STUB: collect the figures and tables into the two dicts save_results expects
figures = {"sml": fig1, "bab_wealth": fig2}
tables = {"capm": capm, "factors": factors,
          "two_factor": two_factor, "spanning": spanning}

save_results(figures=figures, tables=tables, name="ex10")                  # KEEP

## Where this goes next

Two quizzes are open in Moodle until Sunday: the cumulative drill and the transfer
questions. Both are ungraded, and both are exactly the format the exams use.

Everything here was a comparison of alphas across benchmarks, judged by eye. Ten alphas
and ten t-statistics are not ten independent tests, and "the mean absolute alpha fell"
is not a hypothesis test at all. Lecture 11 supplies the statistic that tests all ten at
once, asks how much data a t-statistic of 2.3 really needs, and then turns the spanning
question on the most famous track record in finance — whether the factors you met today
span Berkshire Hathaway.